# 05 - Deploy Qwen2-VL (Gradio demo) — RUN_ID convention

**Mục tiêu:** chạy demo captioning (detail caption) bằng **Gradio** trên Kaggle, thống nhất naming theo pipeline 01–04.

- **RUN_ID**: định danh run và là **tiền tố** (vd: `baseline`, `A`, `B`, `demo`)
- **RUN_KIND**: kiểu chạy (`baseline` | `adapter`)
  - `RUN_KIND="baseline"` ⇒ tự ép `RUN_ID="baseline"`
  - `RUN_KIND="adapter"` ⇒ load adapter theo folder `{RUN_ID}__Qwen2_VL_2B_Instruct/` (hoặc HF repo)


In [11]:
!pip -q install -U gradio transformers accelerate peft qwen-vl-utils

# 1) Notebook configs

In [12]:
import os

RUN_KIND = os.getenv("RUN_KIND", "adapter")    # "baseline" | "adapter"
RUN_ID   = os.getenv("RUN_ID", "A")        # "baseline" | "A" | "B" | "demo"

# Adapter source
ADAPTER_SOURCE   = os.getenv("ADAPTER_SOURCE", "local")  # "local" | "hf"
ADAPTERS_DIR     = os.getenv("ADAPTERS_DIR", "/kaggle/input/vn-textbook-qwen2vl-01-adapters")
HF_ADAPTER_REPO  = os.getenv("HF_ADAPTER_REPO", "")

# Base model (baseline hoặc adapter đều dùng)
BASE_MODEL_ID = os.getenv("BASE_MODEL_ID", "Qwen/Qwen2-VL-2B-Instruct")

# Gradio server configs
GRADIO_SHARE       = os.getenv("GRADIO_SHARE", "1") == "1"
GRADIO_SERVER_NAME = os.getenv("GRADIO_SERVER_NAME", "0.0.0.0")
GRADIO_SERVER_PORT = int(os.getenv("GRADIO_SERVER_PORT", "7860"))


# 2) Helpers / Core logic

In [13]:
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import gradio as gr
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

try:
    from peft import PeftModel
except Exception as e:  # pragma: no cover
    raise RuntimeError(
        "Missing dependency: peft. Please install it (pip install peft)."
    ) from e

try:
    from qwen_vl_utils import process_vision_info
except Exception as e:  # pragma: no cover
    raise RuntimeError(
        "Missing dependency: qwen_vl_utils. Please install qwen-vl-utils "
        "(pip install qwen-vl-utils) or ensure it's available in your environment."
    ) from e


# ----------------------------
# Defaults (can be overridden)
# ----------------------------
DEFAULT_SYSTEM_MESSAGE = (
    "Bạn là một Vision-Language Model hỗ trợ người khiếm thị bằng cách tạo mô tả ảnh bằng tiếng Việt "
    "cho trang sách giáo khoa. Chỉ mô tả những gì thấy trong ảnh, không suy đoán. "
    "Khi OCR, trích toàn bộ chữ nhìn thấy, theo thứ tự trên xuống dưới, trái sang phải."
)

DEFAULT_PROMPT_DETAIL = (
    "Hãy thuyết minh chi tiết trang SGK trong ảnh theo luồng đọc từ trên xuống dưới. "
    "Khi gặp chữ trong ảnh, hãy đưa vào đúng ngữ cảnh và trích nguyên văn trong ngoặc kép. "
    "Không suy luận."
)


def env_bool(name: str, default: bool) -> bool:
    v = os.getenv(name)
    if v is None:
        return default
    return v.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_int(name: str, default: int) -> int:
    v = os.getenv(name)
    if v is None or not v.strip():
        return default
    try:
        return int(v)
    except ValueError:
        return default


def env_float(name: str, default: float) -> float:
    v = os.getenv(name)
    if v is None or not v.strip():
        return default
    try:
        return float(v)
    except ValueError:
        return default


def env_str(name: str, default: str) -> str:
    v = os.getenv(name)
    return default if v is None else v


@dataclass
class RunConfig:
    run_kind: str
    run_id: str
    base_model_id: str

    adapter_source: str  # local|hf
    adapters_dir: Path
    hf_adapter_repo: str

    system_message: str
    prompt_detail: str

    max_new_tokens: int
    do_sample: bool
    temperature: float
    top_p: float


def _safe_read_json(path: Path) -> Dict[str, Any]:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _pick(cfg: Dict[str, Any], keys: Tuple[str, ...], default: Any) -> Any:
    for k in keys:
        if k in cfg and cfg[k] is not None:
            return cfg[k]
    return default


def load_run_config_from_adapter_dir(adapter_root: Path) -> Dict[str, Any]:
    """Load run_config.json if it exists inside an adapter folder."""
    run_cfg = adapter_root / "run_config.json"
    if run_cfg.exists():
        return _safe_read_json(run_cfg)
    return {}


def resolve_adapter_paths(run_id: str, adapters_dir: Path) -> Tuple[Path, Path]:
    """
    Resolve:
      adapter_root: {adapters_dir}/{RUN_ID}__Qwen2_VL_2B_Instruct
      adapter_subdir: adapter_root/adapter (preferred), else adapter_root itself
    """
    adapter_root = adapters_dir / f"{run_id}__Qwen2_VL_2B_Instruct"
    if not adapter_root.exists():
        # fallback: allow user to point directly to root folder
        if adapters_dir.name.startswith(f"{run_id}__"):
            adapter_root = adapters_dir
        else:
            raise FileNotFoundError(
                f"Adapter folder not found: {adapter_root}. "
                f"Set ADAPTERS_DIR correctly or use ADAPTER_SOURCE=hf."
            )

    adapter_subdir = adapter_root / "adapter"
    if adapter_subdir.exists():
        return adapter_root, adapter_subdir
    # Some exports put adapter files at root
    return adapter_root, adapter_root


def build_run_config() -> RunConfig:
    run_kind = env_str("RUN_KIND", "adapter").strip().lower()
    run_id = env_str("RUN_ID", "demo").strip()

    if run_kind not in {"baseline", "adapter"}:
        run_kind = "adapter"

    # Enforce convention
    if run_kind == "baseline":
        run_id = "baseline"

    base_model_id = env_str("BASE_MODEL_ID", "Qwen/Qwen2-VL-2B-Instruct").strip()

    adapter_source = env_str("ADAPTER_SOURCE", "local").strip().lower()
    if adapter_source not in {"local", "hf"}:
        adapter_source = "local"

    adapters_dir = Path(env_str("ADAPTERS_DIR", "/kaggle/input/vn-textbook-qwen2vl-01-adapters/A__Qwen2_VL_2B_Instruct/adapter"))
    hf_adapter_repo = env_str("HF_ADAPTER_REPO", "").strip()

    # Generation params
    max_new_tokens = env_int("MAX_NEW_TOKENS", 1500)
    do_sample = env_bool("DO_SAMPLE", True)
    temperature = env_float("TEMPERATURE", 0.2)
    top_p = env_float("TOP_P", 0.9)

    # Defaults (may be overridden by run_config.json if adapter/local)
    system_message = env_str("SYSTEM_MESSAGE", DEFAULT_SYSTEM_MESSAGE)
    prompt_detail = env_str("PROMPT_DETAIL", DEFAULT_PROMPT_DETAIL)

    # If adapter/local, try load run_config.json to override prompt/gen (best effort)
    if run_kind == "adapter" and adapter_source == "local":
        try:
            adapter_root, _ = resolve_adapter_paths(run_id, adapters_dir)
            rc = load_run_config_from_adapter_dir(adapter_root)
            # Keep backward-compat keys for robustness
            system_message = _pick(
                rc,
                ("system_message", "SYSTEM_MESSAGE", "system", "system_prompt", "SYSTEM_PROMPT"),
                system_message,
            )
            prompt_detail = _pick(
                rc,
                ("prompt_detail", "PROMPT_DETAIL", "prompt", "user_prompt", "PROMPT"),
                prompt_detail,
            )
            gen = rc.get("generation", rc.get("gen", {})) if isinstance(rc, dict) else {}
            if isinstance(gen, dict):
                max_new_tokens = int(gen.get("max_new_tokens", max_new_tokens))
                do_sample = bool(gen.get("do_sample", do_sample))
                temperature = float(gen.get("temperature", temperature))
                top_p = float(gen.get("top_p", top_p))
            base_model_id = str(rc.get("base_model_id", rc.get("model_id", base_model_id)))
        except Exception:
            pass

    return RunConfig(
        run_kind=run_kind,
        run_id=run_id,
        base_model_id=base_model_id,
        adapter_source=adapter_source,
        adapters_dir=adapters_dir,
        hf_adapter_repo=hf_adapter_repo,
        system_message=system_message,
        prompt_detail=prompt_detail,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
    )


def load_model_and_processor(cfg: RunConfig):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    # Base model
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        cfg.base_model_id,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else None,
        trust_remote_code=True,
    )

    # Adapter (optional)
    adapter_note = ""
    if cfg.run_kind == "adapter":
        if cfg.adapter_source == "hf":
            if not cfg.hf_adapter_repo:
                raise ValueError(
                    "ADAPTER_SOURCE=hf but HF_ADAPTER_REPO is empty. "
                    "Set HF_ADAPTER_REPO to your LoRA adapter repo."
                )
            model = PeftModel.from_pretrained(model, cfg.hf_adapter_repo)
            adapter_note = f"HF repo: {cfg.hf_adapter_repo}"
            processor = AutoProcessor.from_pretrained(cfg.base_model_id, trust_remote_code=True)

        else:
            adapter_root, adapter_subdir = resolve_adapter_paths(cfg.run_id, cfg.adapters_dir)
            model = PeftModel.from_pretrained(model, str(adapter_subdir))
            adapter_note = f"Local: {adapter_subdir}"

            # Prefer processor from adapter export if present (optional)
            proc_dir = adapter_root / "processor"
            if proc_dir.exists():
                processor = AutoProcessor.from_pretrained(str(proc_dir), trust_remote_code=True)
            else:
                processor = AutoProcessor.from_pretrained(cfg.base_model_id, trust_remote_code=True)

    else:
        processor = AutoProcessor.from_pretrained(cfg.base_model_id, trust_remote_code=True)

    model.eval()
    return model, processor, device, adapter_note


def _messages(system_message: str, user_prompt: str, image) -> list:
    return [
        {"role": "system", "content": [{"type": "text", "text": system_message}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_prompt},
            ],
        },
    ]


def build_generator(model, processor):
    @torch.inference_mode()
    def generate(image, user_prompt, system_message, max_new_tokens, do_sample, temperature, top_p):
        if image is None:
            return "⚠️ Vui lòng tải ảnh lên.", None

        user_prompt = (user_prompt or "").strip() or DEFAULT_PROMPT_DETAIL
        system_message = (system_message or "").strip() or DEFAULT_SYSTEM_MESSAGE

        msgs = _messages(system_message, user_prompt, image)

        try:
            text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(msgs)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(model.device)

            gen_ids = model.generate(
                **inputs,
                max_new_tokens=int(max_new_tokens),
                do_sample=bool(do_sample),
                temperature=float(temperature),
                top_p=float(top_p),
            )

            gen_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, gen_ids)
            ]
            out = processor.batch_decode(
                gen_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )[0]

            debug = {
                "max_new_tokens": int(max_new_tokens),
                "do_sample": bool(do_sample),
                "temperature": float(temperature),
                "top_p": float(top_p),
                "prompt_used": user_prompt,
                "system_used": system_message,
            }
            return out, debug

        except Exception as e:
            return f"✗ Lỗi khi generate: {e}", None

    return generate


def make_ui(cfg: RunConfig, model, processor, device: str, adapter_note: str):
    generate_fn = build_generator(model, processor)

    css = """
    .container {max-width: 1100px; margin: auto; padding-top: 18px}
    textarea {font-size: 16px !important}
    """

    title = "Vietnamese Textbook Image Captioning (Qwen2-VL)"
    subtitle = f"RUN_KIND: **{cfg.run_kind}** · RUN_ID: **{cfg.run_id}** · Device: **{device}**"
    if cfg.run_kind == "adapter":
        subtitle += f" · Adapter: **{adapter_note or cfg.adapter_source}**"

    with gr.Blocks(css=css, title=title) as demo:
        with gr.Column(elem_classes="container"):
            gr.Markdown(f"#{title}\n\n{subtitle}")

            with gr.Row():
                with gr.Column(scale=4):
                    inp_img = gr.Image(
                        type="pil",
                        label="1) Ảnh đầu vào",
                        sources=["upload", "clipboard"],
                        height=420,
                    )

                    user_prompt = gr.Textbox(
                        label="2) Prompt (detail caption)",
                        value=cfg.prompt_detail,
                        lines=4,
                        max_lines=12,
                    )

                    with gr.Accordion("Tuỳ chọn nâng cao", open=False):
                        system_msg = gr.Textbox(
                            label="System message",
                            value=cfg.system_message,
                            lines=4,
                            max_lines=12,
                        )
                        max_new = gr.Slider(
                            minimum=64,
                            maximum=2048,
                            value=int(cfg.max_new_tokens),
                            step=32,
                            label="max_new_tokens",
                        )
                        do_sample = gr.Checkbox(value=bool(cfg.do_sample), label="do_sample")
                        temperature = gr.Slider(
                            minimum=0.0,
                            maximum=1.5,
                            value=float(cfg.temperature),
                            step=0.05,
                            label="temperature",
                        )
                        top_p = gr.Slider(
                            minimum=0.1,
                            maximum=1.0,
                            value=float(cfg.top_p),
                            step=0.05,
                            label="top_p",
                        )

                    btn = gr.Button("✨ Generate", variant="primary", size="lg")

                with gr.Column(scale=6):
                    out_text = gr.Textbox(
                        label="3) Kết quả mô tả (tiếng Việt)",
                        value="",
                        lines=16,
                        max_lines=120,
                        interactive=False,
                        autoscroll=False,
                    )
                    out_debug = gr.JSON(label="Debug (prompt/gen used)")

            btn.click(
                fn=generate_fn,
                inputs=[inp_img, user_prompt, system_msg, max_new, do_sample, temperature, top_p],
                outputs=[out_text, out_debug],
            )

            gr.Markdown(
                "### Gợi ý\n"
                "- Nếu ảnh là trang SGK có nhiều chữ, hãy tăng `max_new_tokens`.\n"
                "- Nếu muốn kết quả ổn định hơn, tắt `do_sample`.\n"
            )

    return demo




# 3) Build RunConfig (RUN_ID/RUN_KIND)

In [14]:
# Notebook override: set env vars để build_run_config() dùng nhất quán.
os.environ["RUN_KIND"] = str(RUN_KIND)
os.environ["RUN_ID"] = str(RUN_ID)
os.environ["ADAPTER_SOURCE"] = str(ADAPTER_SOURCE)
os.environ["ADAPTERS_DIR"] = str(ADAPTERS_DIR)
os.environ["HF_ADAPTER_REPO"] = str(HF_ADAPTER_REPO)
os.environ["BASE_MODEL_ID"] = str(BASE_MODEL_ID)

cfg = build_run_config()
cfg


RunConfig(run_kind='adapter', run_id='A', base_model_id='Qwen/Qwen2-VL-2B-Instruct', adapter_source='local', adapters_dir=PosixPath('/kaggle/input/vn-textbook-qwen2vl-01-adapters'), hf_adapter_repo='', system_message='Bạn là một Vision-Language Model hỗ trợ người khiếm thị bằng cách tạo mô tả ảnh bằng tiếng Việt cho trang sách giáo khoa. Chỉ mô tả những gì thấy trong ảnh, không suy đoán.Khi OCR, trích toàn bộ chữ nhìn thấy, theo thứ tự trên xuống dưới, trái sang phải', prompt_detail='Hãy thuyết minh chi tiết trang SGK trong ảnh theo luồng đọc từ trên xuống dưới. Khi gặp chữ trong ảnh, hãy đưa vào đúng ngữ cảnh và trích nguyên văn trong ngoặc kép. Không suy luận.', max_new_tokens=1500, do_sample=False, temperature=0.2, top_p=0.9)

# 4) Load model + launch Gradio

In [ ]:
print("⏳ Init deploy...")
print(f"⚙️ RUN_KIND={cfg.run_kind}")
print(f"⚙️ RUN_ID={cfg.run_id}")
print(f"⚙️ ADAPTER_SOURCE={cfg.adapter_source}")
print(f"⚙️ BASE_MODEL_ID={cfg.base_model_id}")

model, processor, device, adapter_note = load_model_and_processor(cfg)
demo = make_ui(cfg, model, processor, device, adapter_note)

demo.launch(
    share=GRADIO_SHARE,
    debug=True,
    server_name=GRADIO_SERVER_NAME,
    server_port=GRADIO_SERVER_PORT,
)


⏳ Init deploy...
⚙️ RUN_KIND=adapter
⚙️ RUN_ID=A
⚙️ ADAPTER_SOURCE=local
⚙️ BASE_MODEL_ID=Qwen/Qwen2-VL-2B-Instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_237/2269923461.py:338: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, title=title) as demo:


* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://eda9818a9dc8bb4994.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
